<a href="https://colab.research.google.com/github/YiruiLi2004/MSSP6070/blob/main/Assignments/Assignment%201/Assignment_1_yiruili.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1: Academic Performance Analysis
This notebook analyzes global assessment scores (`G_SC`) by parental education, socioeconomic group/stratum, gender, academic program, school background, and access to home technology resources.


In [3]:
import os
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import f_oneway, ttest_ind

repo_url = "https://github.com/YiruiLi2004/MSSP6070.git"
repo_path = "/content/MSSP6070"

if not os.path.exists(repo_path):
    subprocess.run(["git", "clone", repo_url], check=True)

assignment_path = "/content/MSSP6070/Assignments/Assignment 1"
os.chdir(assignment_path)

df = pd.read_csv("data_academic_performance 2.csv")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data_academic_performance 2.csv'

## 1. Data Preparation and Quality Check


In [ ]:
# Check basic data quality
print("Dataset shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate student IDs:", df["COD_S11"].duplicated().sum())

missing_values = df.isnull().sum()
print("\nColumns with missing values:")
print(missing_values[missing_values > 0])

print("\nColumn types:")
print(df.dtypes.value_counts())

In [ ]:
# Remove the completely empty column and extra spaces in text categories
df = df.drop(columns=["Unnamed: 9"])
for column in df.select_dtypes(include="object").columns:
    df[column] = df[column].str.strip()

print("Shape after cleaning:", df.shape)
print("Remaining missing values:", df.isnull().sum().sum())


The original file contains 12,411 student records. There are no duplicate rows or duplicate student IDs. `Unnamed: 9` is completely empty and was removed, and extra spaces were stripped from text categories. After cleaning, the dataset has 44 variables and no missing values recognized by pandas. Some responses such as `0`, `Not sure`, and `Ninguno` are still ambiguous, so they are handled separately in the comparisons below.


## 2. Descriptive Statistics Summary


### 2.1 Numerical Variables


In [ ]:
# Descriptive statistics for the main academic score variables
score_columns = ["MAT_S11", "CR_S11", "CC_S11", "BIO_S11", "ENG_S11",
                 "QR_PRO", "CR_PRO", "CC_PRO", "ENG_PRO", "WC_PRO", "G_SC"]

descriptive_summary = pd.DataFrame({
    "N": df[score_columns].count(),
    "Mean": df[score_columns].mean(),
    "Median": df[score_columns].median(),
    "SD": df[score_columns].std(),
    "Min": df[score_columns].min(),
    "Max": df[score_columns].max(),
    "IQR": df[score_columns].quantile(0.75) - df[score_columns].quantile(0.25),
    "Skewness": df[score_columns].skew()
}).round(2)

descriptive_summary


The descriptive statistics give a first view of the center and spread of the academic scores. For the main outcome, the Saber PRO Global Score has a mean of 162.71 and a median of 163. The following sections compare these outcomes across student groups rather than relying on the overall average alone.


### 2.2 Categorical Variables


In [ ]:
# Counts and percentages for the main categorical variables
categorical_vars = ["GENDER", "EDU_FATHER", "EDU_MOTHER", "STRATUM", "SISBEN",
                    "SCHOOL_NAT", "SCHOOL_TYPE", "ACADEMIC_PROGRAM", "INTERNET", "COMPUTER"]

for variable in categorical_vars:
    summary = pd.DataFrame({
        "Count": df[variable].value_counts(),
        "Percent": df[variable].value_counts(normalize=True).mul(100).round(2)
    })
    print(f"\n{variable}")
    display(summary)


### 2.3 Distribution of the Main Outcome


In [ ]:
# Distribution of the Saber PRO Global Score

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

# Histogram
axes[0].hist(df["G_SC"], bins=25, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Global Score")
axes[0].set_ylabel("Number of Students")

# Boxplot
axes[1].boxplot(df["G_SC"], vert=True)
axes[1].set_ylabel("Global Score")
axes[1].set_xticks([1])
axes[1].set_xticklabels(["G_SC"])

plt.tight_layout()
plt.show()


The Global Score distribution is roughly bell-shaped and close to a normal distribution. The mean (162.71) and median (163) are very similar, suggesting that the distribution is fairly symmetric. There are still a few unusually low scores, so it is not perfectly normal.


## 3. Group Comparisons and Further Analysis


### 3.1 Parental Education
**Question:** Does parental education influence outcomes, and if so, how much?


In [ ]:
# Summarize global score by father's and mother's education
father_edu = df.groupby("EDU_FATHER")["G_SC"].agg(["count", "mean", "median", "std"])
mother_edu = df.groupby("EDU_MOTHER")["G_SC"].agg(["count", "mean", "median", "std"])

print("Father's education:")
display(father_edu.sort_values("mean", ascending=False))
print("Mother's education:")
display(mother_edu.sort_values("mean", ascending=False))


In [ ]:
# Compare mean global scores across parental education groups
father_mean = father_edu["mean"].sort_values()
mother_mean = mother_edu["mean"].sort_values()
father_colors = plt.cm.Blues(np.linspace(0.35, 0.85, len(father_mean)))
mother_colors = plt.cm.Purples(np.linspace(0.35, 0.85, len(mother_mean)))

fig, axes = plt.subplots(1, 2, figsize=(17, 7), sharex=True)
father_mean.plot(kind="barh", ax=axes[0], color=father_colors)
mother_mean.plot(kind="barh", ax=axes[1], color=mother_colors)
axes[0].set_title("Father's Education"); axes[1].set_title("Mother's Education")
for ax in axes:
    ax.set_xlabel("Mean Global Score"); ax.set_ylabel("")
for i, value in enumerate(father_mean):
    axes[0].text(value + 0.3, i, f"{value:.1f}", va="center", fontsize=9)
for i, value in enumerate(mother_mean):
    axes[1].text(value + 0.3, i, f"{value:.1f}", va="center", fontsize=9)
plt.suptitle("Mean Global Assessment Score by Parental Education", fontsize=15)
plt.tight_layout()
plt.show()


In [ ]:
# Compare mean global scores across parental education groups

# Remove ambiguous categories
excluded_categories = ["0", "Not sure", "Ninguno"]

father_edu_plot = father_edu[~father_edu.index.isin(excluded_categories)]
mother_edu_plot = mother_edu[~mother_edu.index.isin(excluded_categories)]

father_mean = father_edu_plot["mean"].sort_values()
mother_mean = mother_edu_plot["mean"].sort_values()

father_colors = plt.cm.Blues(np.linspace(0.35, 0.85, len(father_mean)))
mother_colors = plt.cm.Purples(np.linspace(0.35, 0.85, len(mother_mean)))

fig, axes = plt.subplots(1, 2, figsize=(17, 7), sharex=True)

father_mean.plot(kind="barh", ax=axes[0], color=father_colors)
mother_mean.plot(kind="barh", ax=axes[1], color=mother_colors)

axes[0].set_title("Father's Education")
axes[1].set_title("Mother's Education")

for ax in axes:
    ax.set_xlabel("Mean Global Score")
    ax.set_ylabel("")

for i, value in enumerate(father_mean):
    axes[0].text(value + 0.3, i, f"{value:.1f}", va="center", fontsize=9)

for i, value in enumerate(mother_mean):
    axes[1].text(value + 0.3, i, f"{value:.1f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Quantify score differences across selected, clearly defined education levels
print("Father - Primary to Secondary:", round(father_edu.loc["Complete Secundary", "mean"] - father_edu.loc["Complete primary", "mean"], 2))
print("Father - Secondary to Professional:", round(father_edu.loc["Complete professional education", "mean"] - father_edu.loc["Complete Secundary", "mean"], 2))
print("Father - Professional to Postgraduate:", round(father_edu.loc["Postgraduate education", "mean"] - father_edu.loc["Complete professional education", "mean"], 2))
print("Father - Primary to Postgraduate:", round(father_edu.loc["Postgraduate education", "mean"] - father_edu.loc["Complete primary", "mean"], 2))
print("Mother - Primary to Secondary:", round(mother_edu.loc["Complete Secundary", "mean"] - mother_edu.loc["Complete primary", "mean"], 2))
print("Mother - Secondary to Professional:", round(mother_edu.loc["Complete professional education", "mean"] - mother_edu.loc["Complete Secundary", "mean"], 2))
print("Mother - Professional to Postgraduate:", round(mother_edu.loc["Postgraduate education", "mean"] - mother_edu.loc["Complete professional education", "mean"],2))
print("Mother - Primary to Postgraduate:", round(mother_edu.loc["Postgraduate education", "mean"] - mother_edu.loc["Complete primary", "mean"], 2))


In [ ]:
# One-way ANOVA across clearly defined parental education groups
father_analysis = df[~df["EDU_FATHER"].isin(["0", "Not sure", "Ninguno"])].copy()
mother_analysis = df[~df["EDU_MOTHER"].isin(["0", "Not sure", "Ninguno"])].copy()
father_groups = [group["G_SC"].values for name, group in father_analysis.groupby("EDU_FATHER")]
mother_groups = [group["G_SC"].values for name, group in mother_analysis.groupby("EDU_MOTHER")]
father_anova = f_oneway(*father_groups)
mother_anova = f_oneway(*mother_groups)

print("Father's education ANOVA: F =", round(father_anova.statistic, 2), ", p =", father_anova.pvalue)
print("Mother's education ANOVA: F =", round(mother_anova.statistic, 2), ", p =", mother_anova.pvalue)


**Interpretation.** Global scores generally increase with parental education. Students whose fathers had postgraduate education scored about **21.79 points** higher on average than students whose fathers had completed primary education; the corresponding difference for mothers was about **20.30 points**. The ANOVA results show statistically significant differences across both father's and mother's education groups (**p < .001**). These results indicate an association between parental education and academic performance, but they do not establish causation.


### 3.2 Socioeconomic Groups and Strata
**Question:** Which groups/strata perform best, and are there differences between them?


In [ ]:
# Keep valid strata and summarize global scores
stratum_order = ["Stratum 1", "Stratum 2", "Stratum 3", "Stratum 4", "Stratum 5", "Stratum 6"]
df_stratum = df[df["STRATUM"].isin(stratum_order)].copy()
df_stratum["STRATUM"] = pd.Categorical(df_stratum["STRATUM"], categories=stratum_order, ordered=True)
stratum_summary = df_stratum.groupby("STRATUM", observed=True)["G_SC"].agg(["count", "mean", "median", "std"])
stratum_summary


In [ ]:
# Mean global score by socioeconomic stratum
stratum_mean = stratum_summary["mean"]
colors = plt.cm.Greens(np.linspace(0.35, 0.85, len(stratum_mean)))
plt.figure(figsize=(9, 5))
plt.bar(stratum_mean.index, stratum_mean.values, color=colors)
plt.xlabel("Socioeconomic Stratum"); plt.ylabel("Mean Global Score")
for i, value in enumerate(stratum_mean.values):
    plt.text(i, value + 0.5, f"{value:.1f}", ha="center", fontsize=9)
plt.show()

In [ ]:
# Distribution of global scores by stratum
stratum_data = [df_stratum[df_stratum["STRATUM"] == group]["G_SC"] for group in stratum_order]
colors = plt.cm.YlGn(np.linspace(0.35, 0.85, len(stratum_order)))
plt.figure(figsize=(9, 5))
box = plt.boxplot(stratum_data, tick_labels=stratum_order, vert=False, patch_artist=True)
for patch, color in zip(box["boxes"], colors):
    patch.set_facecolor(color)
plt.xlabel("Global Score"); plt.ylabel("Stratum")
plt.show()


In [ ]:
# Test for overall differences across strata
stratum_groups = [group["G_SC"].values for name, group in df_stratum.groupby("STRATUM", observed=True)]
stratum_anova = f_oneway(*stratum_groups)
print("Stratum ANOVA: F =", round(stratum_anova.statistic, 2), ", p =", stratum_anova.pvalue)
print("Stratum 6 - Stratum 1 mean difference:", round(stratum_summary.loc["Stratum 6", "mean"] - stratum_summary.loc["Stratum 1", "mean"], 2))


**Stratum interpretation.** Mean global scores rise steadily from Stratum 1 (**152.35**) to Stratum 6 (**181.51**), a difference of about **29.16 points**. The ANOVA result indicates statistically significant differences across strata (p < .001). This suggests a strong relationship between socioeconomic stratum and academic performance.


In [ ]:
# Exclude the unclear SISBEN category "0" and summarize the remaining groups
df_sisben = df[df["SISBEN"] != "0"].copy()
sisben_summary = df_sisben.groupby("SISBEN")["G_SC"].agg(["count", "mean", "median", "std"])
sisben_summary.sort_values("mean", ascending=False)


In [ ]:
# Mean global score by SISBEN group
sisben_mean = sisben_summary["mean"].sort_values()
colors = plt.cm.Purples(np.linspace(0.35, 0.85, len(sisben_mean)))
plt.figure(figsize=(10, 5))
plt.barh(sisben_mean.index, sisben_mean.values, color=colors)
plt.title("Mean Global Assessment Score by SISBEN Group")
plt.xlabel("Mean Global Score"); plt.ylabel("SISBEN Group")
for i, value in enumerate(sisben_mean.values):
    plt.text(value + 0.3, i, f"{value:.1f}", va="center", fontsize=9)
plt.show()

In [4]:
# Distribution of global scores by SISBEN group
sisben_order = sisben_mean.index.tolist()
sisben_data = [df_sisben[df_sisben["SISBEN"] == group]["G_SC"] for group in sisben_order]
colors = plt.cm.PuRd(np.linspace(0.30, 0.80, len(sisben_order)))
plt.figure(figsize=(10, 5))
box = plt.boxplot(sisben_data, tick_labels=sisben_order, vert=False, patch_artist=True)
for patch, color in zip(box["boxes"], colors):
    patch.set_facecolor(color)
plt.xlabel("Global Score"); plt.ylabel("SISBEN Group")
plt.show()

NameError: name 'sisben_mean' is not defined

In [ ]:
# Test for overall differences across SISBEN groups
sisben_groups = [group["G_SC"].values for name, group in df_sisben.groupby("SISBEN")]
sisben_anova = f_oneway(*sisben_groups)
print("SISBEN ANOVA: F =", round(sisben_anova.statistic, 2), ", p =", sisben_anova.pvalue)
print("Not classified - Level 1 mean difference:", round(sisben_summary.loc["It is not classified by the SISBEN", "mean"] - sisben_summary.loc["Level 1", "mean"], 2))


**SISBEN interpretation.** Students who were not classified by SISBEN had the highest mean global score (**167.47**), while Level 1 students had the lowest (**152.68**), a difference of about **14.79 points**. The ANOVA result shows statistically significant differences across SISBEN groups (p < .001). Because the codebook does not define “not classified” as a specific income level, this category should not automatically be interpreted as the highest socioeconomic group.


### 3.3 Gender
**Question:** How does each gender perform in the global assessment?


#### 3.3.1 Gender Difference

In [ ]:
# Summarize global score by gender
gender_summary = df.groupby("GENDER")["G_SC"].agg(["count", "mean", "median", "std"])
gender_summary


In [ ]:
# Compare mean scores and distributions by gender
gender_order = ["F", "M"]
gender_mean = gender_summary.loc[gender_order, "mean"]
gender_data = [df[df["GENDER"] == group]["G_SC"] for group in gender_order]
bar_colors = plt.cm.cool(np.linspace(0.25, 0.75, 2))
box_colors = plt.cm.cool(np.linspace(0.25, 0.75, 2))

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].bar(gender_mean.index, gender_mean.values, color=bar_colors)
axes[0].set_title("Mean Global Score by Gender"); axes[0].set_xlabel("Gender"); axes[0].set_ylabel("Mean Global Score")
for i, value in enumerate(gender_mean.values):
    axes[0].text(i, value + 0.5, f"{value:.1f}", ha="center")

box = axes[1].boxplot(gender_data, tick_labels=gender_order, patch_artist=True)
for patch, color in zip(box["boxes"], box_colors):
    patch.set_facecolor(color)
axes[1].set_title("Global Score Distribution by Gender"); axes[1].set_xlabel("Gender"); axes[1].set_ylabel("Global Score")
plt.tight_layout()
plt.show()


In [ ]:
# Test for an overall gender difference
female_scores = df[df["GENDER"] == "F"]["G_SC"]
male_scores = df[df["GENDER"] == "M"]["G_SC"]

gender_ttest = ttest_ind(female_scores, male_scores, equal_var=False)

print("Gender t-test:")
print("t-statistic:", round(gender_ttest.statistic, 2))
print("p-value:", gender_ttest.pvalue)


**Interpretation.** Male students had a slightly higher mean global score (**163.69**) than female students (**161.28**), a difference of about **2.41 points**. A Welch’s independent-samples t-test indicated that this difference was statistically significant (**t = -5.78, p < .001**). However, the size of the mean difference was relatively modest compared with the differences observed across parental education and socioeconomic groups.


#### 3.3.2 Gender Differences by Subject

In [ ]:
# Compare academic subject scores by gender
gender_subjects = ["QR_PRO", "CR_PRO", "CC_PRO", "ENG_PRO", "WC_PRO"]
gender_subject_summary = df.groupby("GENDER")[gender_subjects].mean()
gender_subject_summary

In [ ]:
# Compare subject scores and Global Score distributions by gender

gender_order = ["F", "M"]
gender_data = [df[df["GENDER"] == group]["G_SC"] for group in gender_order]

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

# Left: mean Saber PRO subject scores
gender_subject_summary.T.plot(
    kind="bar",
    ax=axes[0],
    color=["#F4A6C1", "#8ECAE6"]
)

axes[0].set_title("Mean Saber PRO Subject Scores by Gender")
axes[0].set_xlabel("Subject")
axes[0].set_ylabel("Mean Score")
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(title="Gender", labels=["Female", "Male"])

for container in axes[0].containers:
    axes[0].bar_label(
        container,
        fmt="%.1f",
        padding=3,
        fontsize=9
    )

# Right: Global Score distribution
box = axes[1].boxplot(
    gender_data,
    tick_labels=["Female", "Male"],
    patch_artist=True
)

box_colors = ["#F4A6C1", "#8ECAE6"]

for patch, color in zip(box["boxes"], box_colors):
    patch.set_facecolor(color)

axes[1].set_title("Global Score Distribution by Gender")
axes[1].set_xlabel("Gender")
axes[1].set_ylabel("Global Score")

plt.tight_layout()
plt.show()

**Interpretation.** The gender differences are not consistent across academic subjects. Male students scored substantially higher in Quantitative Reasoning (**81.20 vs. 71.89**), with a mean difference of about **9.31 points**. In contrast, female students performed better in Written Communication (**56.62 vs. 51.71**), a difference of about **4.91 points**. Male students also had slightly higher mean scores in Critical Reading, Citizen Competencies, and English, but these differences were relatively small, ranging from about **1.03 to 1.63 points**. Welch’s independent-samples t-tests indicated statistically significant gender differences in all five subjects (**p < .05**). However, the largest practical differences were observed in Quantitative Reasoning and Written Communication. These results suggest that the small gender gap in the overall global score may mask more **distinct subject-specific patterns**.

### 3.4 Additional Findings
The following analyses explore academic program, school background, and home technology resources as additional factors that may help inform educational support.


#### 3.4.1 Academic Program


In [ ]:
# Summarize global score by academic program
program_summary = df.groupby("ACADEMIC_PROGRAM")["G_SC"].agg(["count", "mean", "median", "std"])
program_summary.sort_values("mean", ascending=False)


In [ ]:
# Plot programs with at least 10 students(I do this because very small groups can produce unstable means, and cannot reflect the real condition)
program_plot = program_summary[program_summary["count"] >= 10].sort_values("mean")
colors = plt.cm.Oranges(np.linspace(0.30, 0.85, len(program_plot)))

plt.figure(figsize=(10, 8))
plt.scatter(program_plot["mean"], program_plot.index, c=colors, s=70)
for i, value in enumerate(program_plot["mean"]):
    plt.text(value + 0.3, i, f"{value:.1f}", va="center", fontsize=8)
plt.xlim(145, 180)
plt.xlabel("Mean Global Score"); plt.ylabel("Academic Program")
plt.grid(axis="x", linestyle="--", alpha=0.3)
plt.show()


In [ ]:
# ANOVA excluding programs with fewer than 10 students
valid_programs = program_summary[program_summary["count"] >= 10].index
df_program = df[df["ACADEMIC_PROGRAM"].isin(valid_programs)].copy()
program_groups = [group["G_SC"].values for name, group in df_program.groupby("ACADEMIC_PROGRAM")]
program_anova = f_oneway(*program_groups)
print("Academic Program ANOVA: F =", round(program_anova.statistic, 2), ", p =", program_anova.pvalue)


**Interpretation.** Academic performance varies across engineering programs. Among programs with at least 10 students, Chemical Engineering had the highest mean global score (**176.70**), while Electromechanical Engineering had the lowest (**148.62**). The difference should be interpreted cautiously because program sample sizes vary substantially. Programs with only one student were excluded from the plotted comparison and ANOVA because their means are not reliable group estimates.


#### 3.4.2 School Background


In [ ]:
# Summarize school nature and school type
school_nat_summary = df.groupby("SCHOOL_NAT")["G_SC"].agg(["count", "mean", "median", "std"])
school_type_analysis = df[df["SCHOOL_TYPE"] != "Not apply"].copy()
school_type_summary = school_type_analysis.groupby("SCHOOL_TYPE")["G_SC"].agg(["count", "mean", "median", "std"])
print("School nature:")
display(school_nat_summary)
print("School type:")
display(school_type_summary.sort_values("mean", ascending=False))


In [ ]:
# Compare school background groups
school_nat_mean = school_nat_summary["mean"].sort_values()
school_type_mean = school_type_summary["mean"].sort_values()
nat_colors = plt.cm.Reds(np.linspace(0.35, 0.85, len(school_nat_mean)))
type_colors = plt.cm.YlOrBr(np.linspace(0.35, 0.85, len(school_type_mean)))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(school_nat_mean.index, school_nat_mean.values, color=nat_colors)
axes[0].set_xlabel("School Nature"); axes[0].set_ylabel("Mean Global Score")
for i, value in enumerate(school_nat_mean.values):
    axes[0].text(i, value + 0.5, f"{value:.1f}", ha="center")

axes[1].bar(school_type_mean.index, school_type_mean.values, color=type_colors)
axes[1].set_xlabel("School Type"); axes[1].set_ylabel("Mean Global Score")
axes[1].tick_params(axis="x", rotation=20)
for i, value in enumerate(school_type_mean.values):
    axes[1].text(i, value + 0.5, f"{value:.1f}", ha="center")
plt.tight_layout()
plt.show()


In [ ]:
# Test for differences by school nature and school type
private_scores = df[df["SCHOOL_NAT"] == "PRIVATE"]["G_SC"]
public_scores = df[df["SCHOOL_NAT"] == "PUBLIC"]["G_SC"]

school_nat_ttest = ttest_ind(private_scores, public_scores, equal_var=False)
print("School Nature t-test:")
print("t-statistic:", round(school_nat_ttest.statistic, 2))
print("p-value:", school_nat_ttest.pvalue)

school_type_groups = [group["G_SC"].values for name, group in school_type_analysis.groupby("SCHOOL_TYPE")]
school_type_anova = f_oneway(*school_type_groups)
print("School Type ANOVA: F =", round(school_type_anova.statistic, 2), ", p =", school_type_anova.pvalue)

**Interpretation.** Students from private schools had a higher mean global score (**168.22**) than students from public schools (**156.53**), a difference of about **11.69 points**. A Welch’s independent-samples t-test indicated that this difference was statistically significant (**p < .001**). Mean global scores also differed across school types, and the one-way ANOVA indicated statistically significant overall differences among these groups (**p < .001**). These findings show associations between school background and later academic performance, but they do not establish causal effects.


#### 3.4.3 Home Resources and Technology Access


In [ ]:
# Summarize internet and computer access
internet_summary = df.groupby("INTERNET")["G_SC"].agg(["count", "mean", "median", "std"])
computer_summary = df.groupby("COMPUTER")["G_SC"].agg(["count", "mean", "median", "std"])
print("Internet access:")
display(internet_summary)
print("Computer access:")
display(computer_summary)


In [ ]:
# Compare mean global scores by internet and computer access
internet_mean = internet_summary["mean"].sort_values()
computer_mean = computer_summary["mean"].sort_values()
internet_colors = plt.cm.PuBuGn(np.linspace(0.35, 0.80, len(internet_mean)))
computer_colors = plt.cm.YlOrRd(np.linspace(0.30, 0.75, len(computer_mean)))

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].bar(internet_mean.index, internet_mean.values, color=internet_colors)
axes[0].set_title("Mean Global Score by Internet Access"); axes[0].set_xlabel("Internet Access"); axes[0].set_ylabel("Mean Global Score")
for i, value in enumerate(internet_mean.values):
    axes[0].text(i, value + 0.5, f"{value:.1f}", ha="center")

axes[1].bar(computer_mean.index, computer_mean.values, color=computer_colors)
axes[1].set_title("Mean Global Score by Computer Access"); axes[1].set_xlabel("Computer Access"); axes[1].set_ylabel("Mean Global Score")
for i, value in enumerate(computer_mean.values):
    axes[1].text(i, value + 0.5, f"{value:.1f}", ha="center")
plt.tight_layout()
plt.show()


In [ ]:
# Test for differences by internet and computer access
internet_yes = df[df["INTERNET"] == "Yes"]["G_SC"]
internet_no = df[df["INTERNET"] == "No"]["G_SC"]
internet_ttest = ttest_ind(
    internet_yes,
    internet_no,
    equal_var=False
)

print("Internet Access t-test:")
print("t-statistic:", round(internet_ttest.statistic, 2))
print("p-value:", internet_ttest.pvalue)

In [ ]:
computer_yes = df[df["COMPUTER"] == "Yes"]["G_SC"]
computer_no = df[df["COMPUTER"] == "No"]["G_SC"]
computer_ttest = ttest_ind(computer_yes, computer_no, equal_var=False)

print("Computer Access t-test:")
print("t-statistic:", round(computer_ttest.statistic, 2))
print("p-value:", computer_ttest.pvalue)

**Interpretation.** Students with internet access had a mean global score of **164.92**, compared with **154.62** for students without internet access, a difference of about **10.30 points**. Students with a computer also scored higher on average (**164.20** vs. **155.95**), a difference of about **8.24 points**. Both differences are statistically significant (p < .001), although these descriptive comparisons do not show that technology access itself causes higher performance.


## 4. Academic Performance Across Educational Stages

In [ ]:
# Compare academic performance across the two educational stages
fig, axes = plt.subplots(2, 2, figsize=(11, 9))

# Critical Reading
axes[0, 0].scatter(df["CR_S11"], df["CR_PRO"], alpha=0.20, color="#C76D52")
axes[0, 0].set_title("Critical Reading")
axes[0, 0].set_xlabel("Saber 11 Critical Reading Score")
axes[0, 0].set_ylabel("Saber PRO Critical Reading Score")

# Mathematics / Quantitative Reasoning
axes[0, 1].scatter(df["MAT_S11"], df["QR_PRO"], alpha=0.20, color="#3A8D8F")
axes[0, 1].set_title("Mathematics / Quantitative Reasoning")
axes[0, 1].set_xlabel("Saber 11 Mathematics Score")
axes[0, 1].set_ylabel("Saber PRO Quantitative Reasoning Score")

# Citizenship Competencies
axes[1, 0].scatter(df["CC_S11"], df["CC_PRO"], alpha=0.20, color="#8F6FAF")
axes[1, 0].set_title("Citizenship Competencies")
axes[1, 0].set_xlabel("Saber 11 Citizenship Competencies Score")
axes[1, 0].set_ylabel("Saber PRO Citizenship Competencies Score")

# English
axes[1, 1].scatter(df["ENG_S11"], df["ENG_PRO"], alpha=0.20, color="#D39B45")
axes[1, 1].set_title("English")
axes[1, 1].set_xlabel("Saber 11 English Score")
axes[1, 1].set_ylabel("Saber PRO English Score")

plt.tight_layout()
plt.show()

**Interpretation.** The scatterplots suggest **a positive relationship between students' performance at the end of high school and at the end of university**. However, substantial variation remains among students with similar high-school scores. Therefore, these figures show an association between earlier and later academic performance, rather than directly measuring academic growth or value added.

## 5. Summary of Findings

Parental education and socioeconomic stratum show some of the largest score differences in the analysis. School background, academic program, SISBEN classification, and access to internet or computers are also associated with Global Scores. The overall gender difference is small, although the subject-level comparisons show larger differences in Quantitative Reasoning and Written Communication. The scatterplots also suggest continuity between high-school and university performance.
